# 06 - Random Forest Model  
Hotel Booking Demand (Cancellation Prediction)

## Notebook purpose
- Train and tune a Random Forest classifier.
- Save all required Member 4 artifacts for report and comparison.
- Keep output structure consistent with other model notebooks.

## Working directory setup

In [1]:
from pathlib import Path
import os
import subprocess
import sys

try:
    repo_root = subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
    os.chdir(repo_root)
except Exception:
    pass

cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.append(str(cwd))

print("Working directory:", cwd)

Working directory: D:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment


## Imports and artifact paths

In [2]:
import json
import pandas as pd

from src.member4_rf import run_member4_random_forest
from src.config import DEFAULT_DATA_PATH

ART = {
    "models": Path("artifacts/models"),
    "metrics": Path("artifacts/metrics"),
    "plots": Path("artifacts/plots"),
    "reports": Path("artifacts/reports"),
}
for p in ART.values():
    p.mkdir(parents=True, exist_ok=True)

print("Default data path:", DEFAULT_DATA_PATH)

Default data path: data/raw/hotel_bookings.csv


## Dataset check

In [3]:
data_path = Path(DEFAULT_DATA_PATH)
print("Dataset exists:", data_path.exists(), "->", data_path)
if not data_path.exists():
    raise FileNotFoundError("Dataset not found. Put hotel_bookings.csv in data/raw/.")

Dataset exists: True -> data\raw\hotel_bookings.csv


## Train and tune Random Forest

In [4]:
import joblib
import contextlib
from tqdm.auto import tqdm

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

# The notebook output previously logged exactly 540 fits (108 candidates * 5 folds)
with tqdm_joblib(tqdm(desc="RF Hyperparameter Tuning", total=540)):
    
    # For slower machines, reduce max_tune_rows (e.g., 25000) to speed up training
    results = run_member4_random_forest(
        data_path=str(data_path),
        scoring="f1",
        max_tune_rows=40000, 
    )

results

d:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
RF Hyperparameter Tuning:   0%|          | 0/540 [00:00<?, ?it/s]

[data_loader] Dropped duplicates: 31,994 rows
[data_loader] Loaded shape: (87396, 32)
[data_loader] Columns: 32
Fitting 5 folds for each of 108 candidates, totalling 540 fits


RF Hyperparameter Tuning: 100%|██████████| 540/540 [1:30:46<00:00, 10.09s/it]  


{'best_params': {'model__class_weight': 'balanced',
  'model__max_depth': None,
  'model__max_features': 'sqrt',
  'model__min_samples_leaf': 5,
  'model__n_estimators': 400},
 'best_score_cv': 0.7272550195627769,
 'test_metrics': {'accuracy': 0.837929061784897,
  'balanced_accuracy': 0.8289320994115679,
  'precision': 0.6699413995174078,
  'recall': 0.80894901144641,
  'f1': 0.7329122277741115,
  'confusion_matrix': [[10760, 1915], [918, 3887]],
  'roc_auc': 0.9136023988818354,
  'pr_auc': 0.8008738223236871,
  'log_loss': 0.36785055764172614}}

## Load and display saved metrics

In [5]:
with open(ART["metrics"] / "rf_test_metrics.json", "r", encoding="utf-8") as f:
    rf_metrics = json.load(f)

with open(ART["metrics"] / "rf_best_params.json", "r", encoding="utf-8") as f:
    rf_best_params = json.load(f)

print("Best params:")
print(rf_best_params)

metric_order = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "f1", "roc_auc", "pr_auc", "log_loss"
]
rf_table = pd.DataFrame([
    {"model": "random_forest", **{k: rf_metrics.get(k) for k in metric_order}}
])
rf_table

Best params:
{'model__class_weight': 'balanced', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 400}


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss
0,random_forest,0.837929,0.828932,0.669941,0.808949,0.732912,0.913602,0.800874,0.367851


## Feature importance (Top 15)

In [6]:
fi_path = ART["metrics"] / "rf_feature_importance.csv"
if fi_path.exists():
    fi_top15 = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(15)
    fi_top15
else:
    print("Feature importance file not found:", fi_path)

## Artifact verification checklist

In [7]:
expected = [
    ART["models"] / "rf_pipeline.joblib",
    ART["metrics"] / "rf_cv_results.csv",
    ART["metrics"] / "rf_best_params.json",
    ART["metrics"] / "rf_test_metrics.json",
    ART["metrics"] / "rf_feature_importance.csv",
    ART["plots"] / "rf_confusion_matrix.png",
    ART["plots"] / "rf_roc_curve.png",
    ART["plots"] / "rf_pr_curve.png",
    ART["plots"] / "rf_feature_importance.png",
    ART["reports"] / "rf_classification_report.txt",
    ART["reports"] / "rf_notes.md",
]

check_df = pd.DataFrame({
    "artifact": [str(p) for p in expected],
    "exists": [p.exists() for p in expected],
})
check_df

,artifact,exists
0,artifacts\models\rf_pipeline.joblib,True
1,artifacts\metrics\rf_cv_results.csv,True
2,artifacts\metrics\rf_best_params.json,True
3,artifacts\metrics\rf_test_metrics.json,True
4,artifacts\metrics\rf_feature_importance.csv,True
5,artifacts\plots\rf_confusion_matrix.png,True
6,artifacts\plots\rf_roc_curve.png,True
7,artifacts\plots\rf_pr_curve.png,True
8,artifacts\plots\rf_feature_importance.png,True
9,artifacts\reports\rf_classification_report.txt,True


## Results discussion (for report)
- Random Forest captures non-linear interactions in booking behavior.
- Hyperparameter tuning is done with GridSearchCV using F1.
- Results include confusion matrix, ROC/PR curves, and feature importance.
- Use the saved `rf_test_metrics.json` directly in `07_model_comparison.ipynb`.